# Módulo 02 · Segmentación (chunking)

Se comparan tres estrategias para dividir documentos en fragmentos, empezando por la
sección de vacaciones del manual del empleado.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from fundamenta.loaders import load_directory

documents = load_directory(DATA_DIR)
manual = next(doc for doc in documents if doc.source == "manual-del-empleado.md")
start, end = manual.text.index("## Vacaciones"), manual.text.index("## Permisos")
vacations = manual.text[start:end].strip()

print(f"{len(vacations)} caracteres\n")
print(vacations)

462 caracteres

## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.

- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.


In [3]:
def show(chunks: list[str]) -> None:
    for i, chunk in enumerate(chunks):
        print(f"--- fragmento {i} ({len(chunk)} caracteres) ---")
        print(chunk)

## Estrategia 1: tamaño fijo

In [4]:
from fundamenta.chunking import chunk_fixed

show(chunk_fixed(vacations, chunk_size=150))

--- fragmento 0 (150 caracteres) ---
## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.


--- fragmento 1 (150 caracteres) ---
- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 dí
--- fragmento 2 (150 caracteres) ---
as no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5
--- fragmento 3 (12 caracteres) ---
 días extra.


Los cortes parten palabras y frases. La información sobre los días que pueden
trasladarse al año siguiente queda repartida entre dos fragmentos.

### Con solapamiento

Con `overlap=40`, los últimos 40 caracteres de cada fragmento se repiten al inicio del
siguiente: lo que cae en un corte aparece completo en al menos uno de los dos.

In [5]:
show(chunk_fixed(vacations, chunk_size=150, overlap=40))

--- fragmento 0 (150 caracteres) ---
## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.


--- fragmento 1 (150 caracteres) ---
 son días libres para toda la empresa.

- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pued
--- fragmento 2 (150 caracteres) ---
menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cad
--- fragmento 3 (132 caracteres) ---
sarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.


## Estrategia 2: recursiva por separadores

Primero se **divide** por el separador de mayor nivel posible (párrafo, línea, frase,
palabra) hasta que cada pieza cabe en `chunk_size`:

In [6]:
from fundamenta.chunking import split_recursive

for piece in split_recursive(vacations, chunk_size=150):
    print(repr(piece))

'## Vacaciones\n\n'
'Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de\ndiciembre, que son días libres para toda la empresa.\n\n'
'- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días\n'
'  de antelación.\n'
'- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse\n'
'  antes del 31 de marzo.\n'
'- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo\n'
'  de 5 días extra.'


Después se **agrupan** las piezas en orden hasta completar cada fragmento. El
solapamiento se realiza con piezas completas, sin cortar palabras:

In [7]:
from fundamenta.chunking import chunk_recursive

show(chunk_recursive(vacations, chunk_size=150, overlap=40))

--- fragmento 0 (148 caracteres) ---
## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.
--- fragmento 1 (100 caracteres) ---
- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
--- fragmento 2 (124 caracteres) ---
de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
--- fragmento 3 (124 caracteres) ---
antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.


## Estrategia 3: estructura del documento

Un documento Markdown ya está organizado en secciones por sus autores:

In [8]:
from fundamenta.chunking import split_markdown_sections

for path, text in split_markdown_sections(manual.text):
    print(f"{len(text):>5} caracteres   {path}")

  182 caracteres   Manual del empleado de NovaTech
  469 caracteres   Manual del empleado de NovaTech > Bienvenida
  445 caracteres   Manual del empleado de NovaTech > Horario de trabajo
  542 caracteres   Manual del empleado de NovaTech > Teletrabajo
  462 caracteres   Manual del empleado de NovaTech > Vacaciones
  618 caracteres   Manual del empleado de NovaTech > Permisos retribuidos
  201 caracteres   Manual del empleado de NovaTech > Bajas por enfermedad
  216 caracteres   Manual del empleado de NovaTech > Evaluación del desempeño
  258 caracteres   Manual del empleado de NovaTech > Código de conducta


## Integración: `chunk_documents`

Combina las estrategias con los valores por defecto (500 caracteres, 50 de
solapamiento) y conserva los metadatos de cada documento.

In [9]:
from collections import Counter

from fundamenta.chunking import chunk_documents

chunks = chunk_documents(documents, chunk_size=500, overlap=50)
print(f"{len(documents)} documentos -> {len(chunks)} fragmentos\n")
for source, count in Counter(chunk.source for chunk in chunks).items():
    print(f"{source:35} {count:>3} fragmentos")

6 documentos -> 41 fragmentos

beneficios-2026.pdf                   5 fragmentos
faq-soporte-it.txt                    7 fragmentos
manual-del-empleado.md               11 fragmentos
politica-de-gastos-y-viajes.md        9 fragmentos
productos-y-precios.md                9 fragmentos


In [10]:
[chunk] = [c for c in chunks if "23 días laborables" in c.text]
print(chunk.metadata)
print()
print(chunk.text)

{'source': 'manual-del-empleado.md', 'type': 'md', 'title': 'Manual del empleado de NovaTech', 'chunk': 5, 'section': 'Manual del empleado de NovaTech > Vacaciones'}

## Vacaciones

Cada persona tiene 23 días laborables de vacaciones al año, más el 24 y el 31 de
diciembre, que son días libres para toda la empresa.

- Las vacaciones se solicitan en la herramienta interna NovaHR con al menos 15 días
  de antelación.
- Se pueden trasladar al año siguiente hasta 5 días no disfrutados, que deben usarse
  antes del 31 de marzo.
- Por cada 3 años de antigüedad se suma 1 día extra de vacaciones, hasta un máximo
  de 5 días extra.


## Pérdida de contexto

El fragmento con la tabla de permisos no contiene la palabra "permiso"; esa
información está en el título de la sección:

In [11]:
[table] = [c for c in chunks if "Mudanza" in c.text]
print(table.text)
print()
print("¿Contiene 'permiso'?", "permiso" in table.text.lower())
print("Sección:", table.metadata["section"])

| Motivo                                   | Duración              |
|------------------------------------------|-----------------------|
| Matrimonio o registro de pareja de hecho | 15 días naturales     |
| Mudanza                                  | 2 días laborables     |
| Fallecimiento de un familiar directo     | 5 días laborables     |
| Nacimiento o adopción                    | 16 semanas            |
| Asuntos propios                          | 2 días al año         |

¿Contiene 'permiso'? False
Sección: Manual del empleado de NovaTech > Permisos retribuidos


Por ese motivo la sección se guarda en los metadatos y, a partir del módulo 03, se
incorpora al texto que se indexa (`contextualize`).

## Efecto del tamaño de fragmento

In [12]:
print(f"{'chunk_size':>10} {'fragmentos':>10} {'mín':>5} {'media':>6} {'máx':>5}")
for size in (200, 500, 1000, 2000):
    lengths = [len(c.text) for c in chunk_documents(documents, size, size // 10)]
    print(
        f"{size:>10} {len(lengths):>10} {min(lengths):>5} "
        f"{sum(lengths) // len(lengths):>6} {max(lengths):>5}"
    )

chunk_size fragmentos   mín  media   máx
       200         84    49    156   200
       500         41   109    323   498
      1000         30   135    444   988
      2000         28   135    473  1905


A partir de 1.000 caracteres el número de fragmentos apenas varía: las secciones del
Markdown imponen un límite natural. El tamaño óptimo se determina midiendo (módulo 10).